# NormTrace-IHR: Analytical Core Walkthrough

This notebook demonstrates the analytical architecture of NormTrace-IHR.
It shows how the module constructs analysis from structured data objects,
independent of any LLM call.

**Santos-Dominguez, A.B. (2025)**  
NormTrace-IHR (IHR 2005+), Version 1.0.0  
https://doi.org/10.5281/zenodo.XXXXXXX

---

## Contents

1. The IHR article objects: what the selection criterion encodes as data  
2. The four-link enablement chain  
3. The C1 scoring engine applied to a hypothetical country  
4. Schema validation: how the module rejects invalid LLM outputs  
5. Prompt construction from data structures  
6. The administrative law sources as structured data  

In [ ]:
import sys
sys.path.insert(0, '..')

from normtrace.ihr.articles import (
    SELECTED_ARTICLES, Block, SelectionTest, NormRank,
    get_block, get_2024_amendments, get_by_sector, get_by_test
)
from normtrace.ihr.scoring import (
    ChainAssessment, ArticleFinding, BlockScore,
    compute_c1_score, score_block, SituationType, AttentionLevel
)
from normtrace.ihr.schema import validate_and_parse_block, SchemaValidationError
from normtrace.ihr.prompts import build_system_prompt, build_block_prompt
import yaml, json

## 1. The IHR article objects

Each of the 29 selected IHR provisions is a Python object.
The object carries the selection tests that triggered inclusion,
the minimum normative rank required to satisfy the provision,
the domestic law sectors that must be searched, and a direct
reference to the IHR text. These values are not in a prompt;
they are in code, and they can be queried, tested, and cited.

In [ ]:
# Inspect Article 31 -- the provision with the most explicit
# domestic law requirement in the entire IHR

art_31 = SELECTED_ARTICLES['31']

print(f'Article: {art_31.article_id}')
print(f'Block: {art_31.block.value}')
print(f'Title: {art_31.title}')
print(f'Obligation: {art_31.obligation}')
print(f'Selection tests triggered:')
for t in art_31.selection_tests:
    print(f'  - {t.value}')
print(f'Minimum norm rank: {art_31.minimum_norm_rank.value}')
print(f'Sectors to search: {art_31.sectors_to_search}')
print(f'IHR text: {art_31.ihr_text_reference}')
print(f'2024 amendment: {art_31.is_2024_amendment}')
if art_31.notes:
    print(f'Notes: {art_31.notes}')

In [ ]:
# Summary of all 30 article objects across the 7 blocks

print(f'Total article objects: {len(SELECTED_ARTICLES)}')
print(f'(covering 29 provisions; Arts. 36-39 grouped as one object)\n')

for block in Block:
    articles = get_block(block)
    print(f'Block {block.value}: {len(articles)} articles')
    for art_id, art in articles.items():
        amendment = ' [2024]' if art.is_2024_amendment else ''
        tests = ', '.join(t.value for t in art.selection_tests)
        print(f'  Art. {art_id}: {art.title}{amendment}')
        print(f'    Tests: {tests}')
        print(f'    Min rank: {art.minimum_norm_rank.value}')

In [ ]:
# Query: which articles require searching aviation law?
aviation = get_by_sector('aviation')
print(f'Articles requiring aviation law search ({len(aviation)}):')
for art_id, art in aviation.items():
    print(f'  Art. {art_id}: {art.title} [Block {art.block.value}]')

print()

# Query: which articles trigger the COERCIVE_MEASURE test?
coercive = get_by_test(SelectionTest.COERCIVE_MEASURE)
print(f'Articles triggering COERCIVE_MEASURE test ({len(coercive)}):')
for art_id, art in coercive.items():
    print(f'  Art. {art_id}: min rank = {art.minimum_norm_rank.value}')

In [ ]:
# Verification: every article that triggers COERCIVE_MEASURE
# must require at minimum a STATUTE.
# This encodes the reserve of law doctrine as a testable assertion.

violations = [
    (art_id, art)
    for art_id, art in SELECTED_ARTICLES.items()
    if SelectionTest.COERCIVE_MEASURE in art.selection_tests
    and art.minimum_norm_rank != NormRank.STATUTE
]

if violations:
    print('VIOLATIONS FOUND:')
    for art_id, art in violations:
        print(f'  Art. {art_id}: rank = {art.minimum_norm_rank.value}')
else:
    print('All coercive measure articles require STATUTE -- reserve of law criterion satisfied.')

## 2. The four-link enablement chain

For each IHR provision, the LLM assesses the domestic corpus
against four links: norm, actor, authority, enforceability.
The `ChainAssessment` object captures those values and exposes
methods for querying the chain's integrity.

In [ ]:
# Example: a chain for Art. 31 in a country with dispersed coverage
# The health law authorises quarantine (norm: ok) but the actor is
# the health ministry generally (actor: weak), the authority for
# travellers specifically is absent (authority: missing), and
# there is no enforcement mechanism against non-compliant travellers

chain_art31 = ChainAssessment(
    norm='ok',
    actor='weak',
    authority='missing',
    enforceability='missing',
)

print(f'Chain complete: {chain_art31.is_complete()}')
print(f'Weakest link: {chain_art31.weakest_link()}')
print(f'Broken links: {chain_art31.broken_links()}')
print()
print('This chain has a norm but no specific actor, no explicit authority, and')
print('no enforceability. The provision is not normatively enabled despite the')
print('existence of a general quarantine statute.')

## 3. The C1 scoring engine

The C1 score is a pure computation from block scores.
No LLM is involved. The LLM provides the findings;
the module computes the score.

In [ ]:
def make_block_score(block, score):
    return BlockScore(
        block=block, score=score, findings=[],
        gaps=[], articulation_gaps=[], amendment_2024_gaps=[], rationale=''
    )

# Hypothetical: a country with robust health law (A=4, B=4, C=3)
# but no financing mandate (F=1) and no accountability mechanism (F also drives C1.5)

block_scores = {
    Block.A: make_block_score(Block.A, 4),
    Block.B: make_block_score(Block.B, 4),
    Block.C: make_block_score(Block.C, 3),
    Block.D: make_block_score(Block.D, 3),
    Block.E: make_block_score(Block.E, 3),
    Block.F: make_block_score(Block.F, 1),  # no financing mandate
    Block.G: make_block_score(Block.G, 5),
}

# This country's e-SPAR self-report (hypothetical)
espar_score = 3
espar_date = '2024-Q3'

result = compute_c1_score(block_scores, espar_score=espar_score, espar_reference_date=espar_date)

print('C1 NORMATIVE SCORE')
print('='*40)
print(f'  C1.1 Legislation:      {result.c1_1_legislation}/5  (weight 0.30)')
print(f'  C1.2 Financing:        {result.c1_2_financing}/5  (weight 0.20)')
print(f'  C1.3 Coordination:     {result.c1_3_coordination}/5  (weight 0.25)')
print(f'  C1.4 Preparedness:     {result.c1_4_preparedness}/5  (weight 0.15)')
print(f'  C1.5 Accountability:   {result.c1_5_accountability}/5  (weight 0.10)')
print(f'  Weighted total:        {result.total_weighted}/5')
print(f'  Weak-link rule applied:{result.weak_link_applied}')
print()
print('ESPAR COMPARISON')
print('='*40)
print(f'  e-SPAR score:          {result.espar_score} ({result.espar_reference_date})')
print(f'  Delta (e-SPAR - scan): {result.delta:+.2f}')
print(f'  Overreporting:         {result.overreporting_detected()}')
print()
print(result.summary())

In [ ]:
# The weak-link rule: if any indicator = 1, total cannot exceed 2.5
# This encodes the administrative law principle that a broken
# link in the enablement chain renders the whole chain unreliable.

from normtrace.ihr.scoring import apply_weak_link_rule, derive_c1_indicators

indicators = derive_c1_indicators(block_scores)
print('Derived C1 indicators from block scores:')
for k, v in indicators.items():
    print(f'  {k}: {v}')

print()
print(f'Indicator with value 1: c1_2_financing = {indicators["c1_2_financing"]}')
print(f'Without weak-link rule, total would be: {sum(indicators[k] * w for k, w in zip(indicators, [0.30, 0.20, 0.25, 0.15, 0.10])):.2f}')
print(f'With weak-link rule, total is capped at: {result.total_weighted}')

## 4. Schema validation

The validator is the boundary between the LLM and the module.
An LLM output that uses a coverage category not defined in
`articles.py`, omits a required field, or uses an invalid
chain value is rejected before it enters the analysis pipeline.

In [ ]:
# A valid LLM output for Block A
valid_output = {
    'block': 'A',
    'articles': {
        '4': {
            'article_id': '4',
            'coverage': 'articulation_gap',
            'attention_level': 'high',
            'chain': {'norm': 'ok', 'actor': 'weak', 'authority': 'missing', 'enforceability': 'missing'},
            'instruments_searched': ['General Health Law Art. 33', 'Executive Decree 2019-14'],
            'sources_found': [{'name': 'General Health Law', 'article': 'Art. 33', 'url': 'https://example.gov/ley-salud'}],
            'finding_text': 'The health law establishes a general epidemiological surveillance mandate but does not designate a specific national IHR focal point. The 2019 executive decree assigns IHR functions to the Ministry of Health without specifying the focal point role or guaranteeing 24-hour accessibility.',
            'is_2024_gap': False,
        }
    },
    'intersectorality_note': 'Health and migration laws do not reference each other in the context of international notification obligations.',
    'articulation_gaps': ['4'],
    'amendment_2024_gaps': [],
}

parsed, validation = validate_and_parse_block(json.dumps(valid_output))
print(f'Valid output accepted: {validation.valid}')
print(f'Warnings: {validation.warnings}')

In [ ]:
# An invalid LLM output: coverage value not in the defined set
invalid_output = dict(valid_output)
invalid_output['articles'] = dict(valid_output['articles'])
invalid_finding = dict(valid_output['articles']['4'])
invalid_finding['coverage'] = 'critical_gap'  # not a valid SituationType
invalid_finding['chain']['norm'] = 'partially'  # not a valid chain value
invalid_output['articles']['4'] = invalid_finding

try:
    validate_and_parse_block(json.dumps(invalid_output))
    print('Should not reach here')
except SchemaValidationError as exc:
    print(f'Correctly rejected with {len(exc.errors)} error(s):')
    for err in exc.errors:
        print(f'  - {err}')

## 5. Prompt construction from data structures

Prompts are not hardcoded. They are assembled from the article objects.
The sectors to search in the corpus discovery prompt are derived
from the `sectors_to_search` attribute of all 30 article objects.
The article descriptions in the block prompts are generated from
the article objects at call time.

In [ ]:
# The sectors in the corpus discovery prompt are derived from articles.py,
# not written by hand in the prompt function.

all_sectors = sorted(set(
    sector
    for article in SELECTED_ARTICLES.values()
    for sector in article.sectors_to_search
))

print('Sectors derived from article objects:')
for s in all_sectors:
    articles_requiring = [art_id for art_id, art in SELECTED_ARTICLES.items() if s in art.sectors_to_search]
    print(f'  {s}: {articles_requiring}')

In [ ]:
# Block A prompt: article descriptions are generated from objects
# The prompt changes if the article objects change -- they are the source of truth

corpus = [
    {'name': 'General Health Law', 'type': 'statute', 'sector': 'health',
     'url': 'https://example.gov/ley-salud', 'last_reform': 'DOF 15-01-2026'},
    {'name': 'Migration Law', 'type': 'statute', 'sector': 'migration',
     'url': 'https://example.gov/ley-migracion', 'last_reform': 'DOF 27-05-2024'},
]

prompt = build_block_prompt(Block.A, 'Example Country', corpus, language='en')

print(f'Block A prompt length: {len(prompt)} characters')
print(f'Contains Art. 4bis: {"4bis" in prompt}')
print(f'Contains 2024 amendment flag: {"2024 AMENDMENT" in prompt}')
print(f'Contains corpus instruments: {"General Health Law" in prompt}')
print()
print('First 800 characters:')
print(prompt[:800])

## 6. Administrative law sources as structured data

The theoretical grounding of the selection criterion is in a YAML file
(`normtrace/references/legal_theory.yaml`) that can be read programmatically.
Each source has a `doctrine` field listing which analytical concepts it supports,
and a `relevance` field explaining how it grounds the NormTrace criterion.
These can be queried just like the article objects.

In [ ]:
with open('../normtrace/references/legal_theory.yaml') as f:
    sources = yaml.safe_load(f)['sources']

print(f'Total sources: {len(sources)}')
print()

# Sources by legal tradition
traditions = {}
for src in sources:
    for trad in src.get('tradition', []):
        traditions.setdefault(trad, []).append(src['id'])

print('Sources by legal tradition:')
for trad, ids in sorted(traditions.items()):
    print(f'  {trad}: {ids}')

In [ ]:
# Query: which sources ground the reserve_de_loi / reserva_de_ley doctrine?
reserve_sources = [
    src for src in sources
    if any('reserva' in d or 'reserve' in d or 'Vorbehalt' in d
           for d in src.get('doctrine', []))
]

print('Sources grounding the reserve of law doctrine:')
for src in reserve_sources:
    authors = ', '.join(
        f"{a.get('family', a.get('name', ''))}" for a in src['author']
    )
    print(f'  {authors} ({src["year"]}): {src["title"]}')
    print(f'    Relevance: {src["relevance"][:120]}...')

In [ ]:
# The connection between doctrine and articles is explicit:
# COERCIVE_MEASURE test -> reserve of law -> statute required
# This chain is traceable from data to theory to code.

print('Analytical traceability chain for Art. 31 (quarantine of travellers):')
print()
print('IHR TEXT')
print(f'  {SELECTED_ARTICLES["31"].ihr_text_reference}')
print()
print('SELECTION TEST TRIGGERED')
print(f'  {[t.value for t in SELECTED_ARTICLES["31"].selection_tests]}')
print()
print('MINIMUM NORM RANK REQUIRED')
print(f'  {SELECTED_ARTICLES["31"].minimum_norm_rank.value}')
print()
print('DOCTRINE GROUNDING THE REQUIREMENT')
for src in reserve_sources:
    authors = ', '.join(a.get('family', a.get('name', '')) for a in src['author'])
    print(f'  {authors} ({src["year"]}): {src["title"]}')

---

## Citation

If you use NormTrace-IHR in published research, please cite:

> Santos-Dominguez, A.B. (2025). *NormTrace-IHR (IHR 2005+): A systematic tool
> for assessing the domestic legal implementation of the International Health
> Regulations* (Version 1.0.0). Zenodo. https://doi.org/10.5281/zenodo.XXXXXXX

See `normtrace/references/references.bib` for BibTeX entries of all
administrative law sources cited in the analytical framework.